# 任务 02 演示：D1 × E2

本 notebook 严格按 **业务问题 → setting → support → POINT 推导 → IdentificationResult** 的顺序展开。

这是一个**识别（identification）**作业：不读模拟器的结果真值，不估计任何数字，也不碰
Estimation 与 Policy。唯一被计算的量，是"这次查询下的总体支撑概率"

```
π_i(t, s) = P(T_i = t, S_i = s | A, X)
```

**业务问题。** 一家公司给共享客户线索的销售员开放 AI 销售助手。一名销售员的业绩同时取决于两件事：
**自己**有没有助手（`T_i`），以及**与他合作的同事中有多少人也在用**（他的**暴露**）。我们关心的
因果量是 **arm mean**（处理组合均值）`μ_i(t, s)`，即"自己处理为 `t`、暴露状态为 `s` 时 i 的平均
业绩"。本任务中自己的处理有 2 种取值，暴露有 **3 档**。

## 1. Setting：任务 02 相对 baseline 改了什么

**分配规则 D1** —— 每人概率不同的独立伯努利：

```
T_i ~ Bernoulli(p_i)，各人独立，0 < p_i < 1，p_i 在处理前已知
```

**暴露定义 E2** —— 按"被处理的邻居占比"分三档：

```
K_i(u) = Σ_{j∈N(i)} u_j          E_i(u) = K_i(u) / d_i

s = 0（低）   若 E_i(u) < 1/3
s = 1（中）   若 1/3 ≤ E_i(u) < 2/3
s = 2（高）   若 E_i(u) ≥ 2/3
```

因为 `K_i` 取整数，两条分界线要先换算成整数区间：设 `k1 = ⌈d/3⌉`、`k2 = ⌈2d/3⌉`，则

```
s = 0 → K_i ∈ [0, k1 − 1]      s = 1 → K_i ∈ [k1, k2 − 1]      s = 2 → K_i ∈ [k2, d]
```

其余部分沿用共享 baseline：处理前固定的网络 `G = (V, A)`、只作为条件背景的 `X`、
一致性（consistency）、整体随机化（full-vector randomization），以及 support 的定义。

In [ ]:
import sys
from pathlib import Path
import numpy as np

# 定位 student_work/task02 目录：从当前目录或仓库根目录启动都能找到
TASK_DIR = next((c.resolve() for c in (Path.cwd(), Path.cwd() / "student_work" / "task02")
                 if (c / "support.py").exists()), None)
assert TASK_DIR is not None, "请在 student_work/task02 或仓库根目录下运行本 notebook"
sys.path.insert(0, str(TASK_DIR))

import support as sp
import rule as rl

print("任务目录 :", TASK_DIR)
print("任务编号 : 任务 02 = D1（异质独立伯努利）× E2（三档暴露）")

In [ ]:
DATA = (TASK_DIR.parents[1] / "pfn_pipeline" / "_internal" / "estimation" / "data"
        / "fixed_er_N300_p0.02_seed12345.json")

if DATA.exists():
    adj = sp.load_fixed_er_adjacency(DATA)
    source = "老师给的固定 ER300 网络：" + DATA.name
else:
    upper = np.triu(np.random.default_rng(12345).random((300, 300)) < 0.02, k=1)
    adj = (upper | upper.T).astype(np.int8)
    source = "本地生成的 ER 图 N=300（没找到固定网络文件）"

n_units = adj.shape[0]
degrees = adj.sum(axis=1).astype(int)
ps = np.random.default_rng(2026).uniform(0.2, 0.8, size=n_units)      # p_i 是本 setting 自己定的设计参数

print("图来源   :", source)
print("节点 / 边:", n_units, "/", int(adj.sum()) // 2)
print("度数范围 : %d - %d" % (degrees.min(), degrees.max()))
print("p_i 范围 : %.3f - %.3f" % (ps.min(), ps.max()))
print("孤立节点数（E2 不适用）:", int((degrees == 0).sum()))
print("度数为 1 的节点数（medium 档不可达）:", int((degrees == 1).sum()))

## 2. 总体支撑概率 π_i(t, s)

在 D1 下，各人的处理指示变量互相独立，而且一个人永远不是自己的邻居，所以 `T_i` 与 `S_i` 独立，
支撑概率可以直接拆成两项相乘：

```
π_i(t, s) = P(T_i = t) · P(S_i = s)
          = [ t·p_i + (1 − t)(1 − p_i) ] · Σ_{k ∈ R_s(d_i)} P(K_i = k)
```

`K_i` 是 `d_i` 个**独立但成功概率各不相同**的伯努利变量之和，这个分布叫**泊松二项
（Poisson-binomial）**，不是普通二项分布，由 `support.py` 里的动态规划精确计算。注意：
只有**邻居**的 `p_j` 会进入这个分布，自己的 `p_i` 只出现在第一个因子里。

In [ ]:
def show_support(adjacency, probabilities, unit):
    pi = sp.support_D1_E2(adjacency, probabilities, unit)
    d = sp.degree(adjacency, unit)
    print("节点 %d：d_i = %d，N(i) = %s"
          % (unit, d, list(map(int, sp.neighbours(adjacency, unit)))))
    for s in sp.E2_LEVELS:
        lo, hi = sp.e2_integer_ranges(d)[s]
        label = "[%d, %d]" % (lo, hi) if hi >= lo else "空区间（无支撑）"
        print("   s = %d：P(S_i = %d) = %.6f    K_i 落在 %s"
              % (s, s, sum(pi[(t, s)] for t in (0, 1)), label))
    print("        " + "".join("%16s" % ("s = " + str(s)) for s in sp.E2_LEVELS))
    for t in (0, 1):
        print("   t = %d " % t + "".join("%16.6f" % pi[(t, s)] for s in sp.E2_LEVELS))
    print("   六个 arm 之和 = %.12f" % sum(pi.values()))
    print()

supported_units = [i for i in range(n_units) if degrees[i] >= 2]
print("== 正常例：度数 >= 2 的节点 ==")
show_support(adj, ps, supported_units[0])

# 手造小图：7 号节点只有一个邻居，所以它的 medium 档永远不可达
SMALL = np.zeros((10, 10), dtype=np.int8)
for a, b in [(0, 1), (0, 2), (0, 3), (1, 4), (2, 5), (3, 6), (4, 7), (5, 8), (6, 9)]:
    SMALL[a, b] = SMALL[b, a] = 1
SMALL_PS = np.linspace(0.25, 0.75, 10)
DEGREE_ONE_UNIT = 7
assert sp.degree(SMALL, DEGREE_ONE_UNIT) == 1
print("== 零支撑例：手造小图上度数为 1 的节点 ==")
show_support(SMALL, SMALL_PS, DEGREE_ONE_UNIT)

zero_support = [(i, int(degrees[i])) for i in range(n_units)
                if any(v == 0.0 for v in sp.support_D1_E2(adj, ps, i).values())]
print("至少有一个 arm 支撑概率为 0 的节点：", len(zero_support), zero_support[:10])
print("因为 0 < p_j < 1，所以 π_i(t, s) > 0  ⟺  该档的整数区间非空；")
print("对 E2 而言：d_i >= 2 → 六个 arm 全有支撑；d_i = 1 → 两个 medium arm 没有支撑。")

## 3. 为什么有支撑的 arm 是 POINT

对任意满足 `π_i(t, s) > 0` 的 `(i, t, s)`，把条件事件按它包含的分配向量拆开，再代入两个假设
（这一步与任务书第 3 节完全一致）：

```
E[Y_obs_i | A, X, T_i = t, S_i = s]
  = Σ_{u_-i : S_i((t, u_-i)) = s} E[Y_obs_i | A, X, T = (t, u_-i)] · w(u_-i)   （全概率公式）
  = Σ_{u_-i : S_i((t, u_-i)) = s} E[Y_i((t, u_-i)) | A, X] · w(u_-i)          （随机化 + 一致性）
  = μ_i(t, s)                                                                （目标量的定义）
```

其中 `w(u_-i) = q((t, u_-i)) / π_i(t, s)`。右边每一项都是总体可观测分布 `P(T, Y_obs | A, X)` 的
泛函，所以"观测分布完全相同的两个世界"必然给出一致的因果结论：

```
μ_i(t, s) = E[ Y_obs_i | A, X, T_i = t, S_i = s ]        （式 3）
```

当 `π_i(t, s) = 0` 时，条件事件 `{T_i = t, S_i = s}` 的概率为零，条件期望**根本不存在**（不是 0，
也不是 NaN），所以式(3) **不能**使用——此时规则必须明确拒绝。

In [ ]:
unit = supported_units[0]
print("--- 有支撑的查询：节点 %d，d_i = %d ---" % (unit, degrees[unit]))
arm = rl.identify_arm(adj, ps, unit, 1, 2)
print("状态     :", arm.status)
print("支撑概率 : %.6f" % arm.support)
print("识别泛函 :", arm.functional, "  ← 给的是公式，不是数字")
print()

print("--- 零支撑查询：节点 %d，d_i = 1 ---" % DEGREE_ONE_UNIT)
try:
    rl.identify_arm(SMALL, SMALL_PS, DEGREE_ONE_UNIT, 1, 1)
except rl.ZeroSupportRefusal:
    print("已拒绝   : medium 档的整数区间为空，该 arm 不可达，式(3) 无法使用")
refused = rl.try_identify_arm(SMALL, SMALL_PS, DEGREE_ONE_UNIT, 1, 1)
print("状态     :", refused.status, "| 支撑 =", refused.support, "| 泛函 =", repr(refused.functional))
ok = rl.try_identify_arm(SMALL, SMALL_PS, DEGREE_ONE_UNIT, 1, 2)
print("其它档仍然可用，例如 (t=1, s=2)：", ok.status, "| π = %.6f" % ok.support)

try:
    rl.identify_arm(adj, ps, unit, 1, 2, assumptions_admitted=False)
except rl.UnresolvedAssumptionsError:
    print()
    print("假设还没确认时，同样拒绝——不能只凭设计就宣称可识别")

## 4. IdentificationResult

`rule.identify_all_arms` 一次跑完某个节点的 6 个 arm，逐个报告：总体支撑概率、POINT 状态、
识别泛函，或者拒绝的理由。

因为 E2 有三档，而**中等档在小度节点上可能不可达**，所以本 setting 的效应对比特意只声明在
低档和高档上，保证任何对比都不会碰到那个可能无支撑的档：

```
τ_i^Direct    = μ_i(1, 0) − μ_i(0, 0)
τ_i^Spillover = μ_i(1, 2) − μ_i(1, 0)
τ_i^Total     = μ_i(1, 2) − μ_i(0, 0)
```

In [ ]:
unit = supported_units[0]
result = rl.identify_all_arms(adj, ps, unit)

print("任务编号 : 任务 02 = D1（异质独立伯努利）× E2（三档暴露）")
print("结果状态 :", result.status, "| 三个假设均已确认:", result.assumptions_admitted)
print("需要的三个假设（原文见 rule.py）：")
print("   1. 一致性：实际分配为 u 时，观测结果等于潜在结果 Y_i(u)")
print("   2. 整体随机化：给定 A, X 时，整个分配向量 T 与全部潜在结果独立（来自实验协议）")
print("   3. 该查询的正总体支撑：π_i(t, s) > 0")
print()
for arm in result.arms:
    detail = arm.functional if arm.identified else "无支撑，已拒绝"
    print("  (t = %d, s = %d)  π = %s  %-8s %s"
          % (arm.treatment, arm.exposure,
             "%.6f" % arm.support if arm.support is not None else "不适用", arm.status, detail))

total_supported = total_arms = 0
refused_units = []
for i in range(n_units):
    unit_result = rl.identify_all_arms(adj, ps, i)
    total_supported += len(unit_result.supported_arms)
    total_arms += len(unit_result.arms)
    if unit_result.refused_arms:
        refused_units.append(i)

print()
print("全图统计：%d 个 arm 有支撑，共 %d 个 arm（6 × N = %d）"
      % (total_supported, total_arms, 6 * n_units))
print("含有被拒绝 arm 的节点:", len(refused_units), "（就是那些度数为 1 的节点）")
print()
print("有支撑的 arm 被点识别为一个可观测量：", rl.POINT_FUNCTIONAL_TEMPLATE)
print("被拒绝的 arm 一律报告 REFUSED 且不给泛函——绝不返回 0、NaN，也不退回 PARTIAL。")